In [0]:
%run "/Workspace/Repos/Git Repo/dwh_sql_repo/Medallion Architecture/Generic Framework and 3layer data load/Generic function"

In [0]:
from pyspark.sql import SparkSession
sparkss = spark_session("Logistcs Data Project")

SRC = "/Volumes/datalake_prod/datalake_prod_schema/datalake/"
BRONZE = "/Volumes/datalake_prod/datalake_prod_schema/bronze/"

bronze_path = f"""{BRONZE}/logistics_staff"""
bronze_path1 = f"""{BRONZE}/logistics_Geo"""
bronze_pat2 = f"""{BRONZE}/logistics_shipment"""

staff_delta = read_delta_df(sparkss,bronze_path)
geo_delta = read_delta_df(sparkss,bronze_path1)
shipment_delta = read_delta_df(sparkss,bronze_pat2)


In [0]:
silver_staff = standardize_staff(staff_delta)

silver_geotag=scrub_geotag(geo_delta).distinct()

silver_filtered_shipment=shipment_delta.where("shipment_weight_kg>0")
silver_shipments = (silver_filtered_shipment
    .transform(standardize_shipments)
    .transform(enrich_shipments)
    .transform(split_columns))

In [0]:

SILVER = "/Volumes/datalake_prod/datalake_prod_schema/silver"
SILVERDB = "datalake_prod.datalake_prod_schema"

write_file(silver_staff,f"{SILVER}/staff",mode="overwrite",format="delta")
write_file(silver_geotag,f"{SILVER}/geotag",mode="overwrite",format="delta")
write_file(silver_shipments,f"{SILVER}/shipments",mode="overwrite",format="delta")

write_table(silver_staff,f"{SILVERDB}.silver_staff", mode="overwrite")
write_table(silver_geotag,f"{SILVERDB}.silver_geotag", mode="overwrite")
write_table(silver_shipments,f"{SILVERDB}.silver_shipments", mode="overwrite")